# Comparaison des pipelines de classification — Branche A vs Branche B

**Auteur** : Coletta Moutsiha 
**Date** : 29 avril 2026

Ce notebook réalise l'**analyse statistique comparative** des trois pipelines de
classification des maladies de la tomate dans le cadre du pipeline phase 1 :

| Pipeline    | Branche | Description                                |
|-------------|---------|--------------------------------------------|
| `baseline`  | **A**   | ResNet50 sur image complète (sans pré-traitement) |
| `blur_light`| A′      | ResNet50 entraîné avec un GaussianBlur léger sur le train ; **val et test inchangé** |
| `sam3_cnn`  | **B**   | ResNet50 sur image segmentée par SAM3 (feuille isolée) |

Chaque pipeline a été entraîné avec **8 seeds** (16, 42, 123, 698, 1373, 1979, 2026, 5245) sur le dataset
*Tomato_final* (22 265 images, 12 classes), avec un protocole identique :
ResNet50 ImageNet → entête remplacée (Dropout 0,3 + Linear) → entraînement en 2 phases
(tête seule 5 ép., puis fine-tuning layer4 + tête).

L'analyse suit **4 niveaux** :

1. **Statistiques descriptives** : moyennes, écarts-types, IC95 % bootstrap
2. **Comparaisons appariées** : Wilcoxon signé sur paires de pipelines + correction multi-tests
3. **Analyse par classe** : focus sur les classes difficiles (`verticillium_wilt`,
   `leaf_blight`, `yellow_leaf_curl_virus`)
4. **Analyse résiduelle** : matrices de confusion agrégées + accord/désaccord entre pipelines
   (re-prédit sur le test set avec les checkpoints `best_*.pth`)


## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import stats_analysis as sa

# Style coherent pour toutes les figures
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
})

PIPE_COLORS = {p: cfg["color"] for p, cfg in sa.PIPELINES.items()}
PIPE_LABELS = {p: cfg["label"] for p, cfg in sa.PIPELINES.items()}

# A ADAPTER : chemin du dossier qui contient tous les results_*_seed_*
BASE_DIR = Path("/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/classification_tomato/3pipelinetest")  # ajuste selon ton arborescence locale

# Dossier de sortie pour les figures du rapport
FIG_DIR = Path("./figures_rapport3")
FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = Path("./tableaux_rapport3")
TAB_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("./predictions_cache")
CACHE_DIR.mkdir(exist_ok=True)


In [ ]:
# Charger les 24 runs (3 pipelines x 8 seeds)
results = sa.load_all_results(BASE_DIR, seeds=sa.SEEDS)

print(f"Pipelines charges : {list(results.keys())}")
for pipe, runs in results.items():
    print(f"  {pipe:12s} : {len(runs)} seeds  -> "
          f"acc moyen = {np.mean([r.test_accuracy for r in runs])*100:.2f}%, "
          f"f1 moyen = {np.mean([r.test_macro_f1 for r in runs])*100:.2f}%")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
from torchvision import transforms

# ── 1. Charger une image de ton dataset ──────────────────────────────────────
# Adapte le chemin vers une image quelconque de Tomato_final
img_path = "/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/data/dataset_utilise/tomato_final/train/tomato_late_blight/IMG-20240805-WA0025.jpg"
img = Image.open(img_path).convert("RGB")

# ── 2. Reconstruire les transforms (sans ToTensor/Normalize pour visualiser) ─
# On récupère tes transforms mais on s'arrête avant ToTensor pour garder du PIL

base_train_ops = [
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
]

blur_op = transforms.RandomApply(
    [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))],
    p=1.0,  # ← p=1.0 pour FORCER le flou et le voir à coup sûr
)

# ── 3. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Ligne 1 : Baseline (sans flou)
baseline_tf = transforms.Compose(base_train_ops)
axes[0, 0].imshow(img)
axes[0, 0].set_title("Originale")
axes[0, 0].axis("off")

for i in range(1, 4):
    augmented = baseline_tf(img)
    axes[0, i].imshow(augmented)
    axes[0, i].set_title(f"Baseline — augmentation {i}")
    axes[0, i].axis("off")

# Ligne 2 : Blur light (avec flou forcé pour visualisation)
blur_tf = transforms.Compose(base_train_ops + [blur_op])
axes[1, 0].imshow(img)
axes[1, 0].set_title("Originale")
axes[1, 0].axis("off")

for i in range(1, 4):
    augmented = blur_tf(img)
    axes[1, i].imshow(augmented)
    axes[1, i].set_title(f"Blur light — augmentation {i}")
    axes[1, i].axis("off")

plt.suptitle("Comparaison des augmentations : Baseline (haut) vs Blur light (bas)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as F

img_path = "/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/data/dataset_utilise/tomato_final/train/tomato_late_blight/IMG-20240805-WA0025.jpg"
img = Image.open(img_path).convert("RGB")

# Recadrer pour avoir une taille fixe
img_resized = transforms.Resize((224, 224))(img)

# Tester plusieurs sigmas
sigmas = [0.0, 0.5, 1.0, 2.0, 4.0]

fig, axes = plt.subplots(1, len(sigmas), figsize=(20, 4))

for ax, sigma in zip(axes, sigmas):
    if sigma == 0.0:
        blurred = img_resized
        title = "Original (σ=0)"
    else:
        blurred = F.gaussian_blur(img_resized, kernel_size=5, sigma=sigma)
        title = f"σ = {sigma}"
    ax.imshow(blurred)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Effet du sigma sur le flou gaussien", fontsize=14)
plt.tight_layout()
plt.show()

## 1. Statistiques descriptives globales

On commence par regarder la **performance globale** de chaque pipeline, agrégée sur les 8 seeds.
Trois métriques sont reportées : `test_accuracy`, `test_macro_f1` (métrique principale, robuste
au déséquilibre de classes) et `test_loss`. Pour chacune, on calcule la moyenne, l'écart-type,
les valeurs extrêmes, et un **intervalle de confiance bootstrap à 95 %** sur la moyenne
(10 000 ré-échantillonnages).


In [ ]:
desc = sa.descriptive_stats(results)
desc_display = desc.copy()
for col in ("mean", "std", "min", "max", "ci95_low", "ci95_high"):
    desc_display[col] = desc_display[col].apply(lambda x: f"{x*100:.3f}" if abs(x) < 5 else f"{x:.3f}")
desc_display


In [ ]:
# Tableau formate pour le rapport (Macro-F1)
tbl_f1 = sa.format_mean_std_table(desc, metric="test_macro_f1")
tbl_acc = sa.format_mean_std_table(desc, metric="test_accuracy")

print("=== Macro-F1 ===")
print(tbl_f1.to_string(index=False))
print("\n=== Accuracy ===")
print(tbl_acc.to_string(index=False))

# Sauvegarde CSV pour conversion LaTeX
tbl_f1.to_csv(TAB_DIR / "table_descriptif_macro_f1.csv", index=False)
tbl_acc.to_csv(TAB_DIR / "table_descriptif_accuracy.csv", index=False)
desc.to_csv(TAB_DIR / "descriptif_complet.csv", index=False)


### Visualisation : boxplots + points par seed

Le boxplot synthétise la distribution sur les 8 seeds, et les points individuels (`stripplot`) garantissent
la **transparence** : on voit chaque seed, donc le lecteur peut juger lui-même la stabilité du pipeline.


In [ ]:
long = sa.long_format_metrics(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = list(sa.PIPELINES.keys())

for ax, metric, title in zip(
    axes,
    ["test_macro_f1", "test_accuracy"],
    ["Macro-F1 sur le test set", "Accuracy sur le test set"],
):
    sub = long[long["metric"] == metric]

    sns.boxplot(
        data=sub,
        x="pipeline",
        y="value",
        hue="pipeline",
        order=order,
        palette=PIPE_COLORS,
        ax=ax,
        width=0.5,
        fliersize=0,
        legend=False,
    )

    sns.stripplot(
        data=sub,
        x="pipeline",
        y="value",
        order=order,
        color="black",
        size=6,
        ax=ax,
        alpha=0.7,
    )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric.replace("_", " "))
    
    # Annotation moyenne ± std au-dessus de chaque boxplot
    for i, p in enumerate(order):
        vals = sub[sub["pipeline"] == p]["value"].to_numpy()
        ax.text(
            i,
            vals.max() + 0.003,
            f"{vals.mean()*100:.2f}\n±{vals.std(ddof=1)*100:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

plt.suptitle("Distribution des métriques sur 8 seeds par pipeline", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "boxplots_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

**Lecture.** L'écart-type inter-seeds reflète la stabilité d'entraînement de chaque pipeline.
Une variance faible (< 0,5 pt sur Macro-F1) indique que les conclusions ne dépendent pas du seed.


## 2. Comparaisons appariées

On compare maintenant les pipelines **deux à deux**. Étant donné que les mêmes seeds sont
appliquées aux trois pipelines, on est dans un design **apparié** (chaque seed crée un trio
de mesures appariées) : c'est ce qu'il faut exploiter pour gagner en puissance statistique.

**Tests utilisés** :

- **Wilcoxon signé apparié** (test principal) : non-paramétrique, robuste à petits échantillons,
  ne suppose pas la normalité des différences.
- **Test t apparié** (sanity-check) : reporté pour information.
- **Cohen's d_z** : taille d'effet pour mesures appariées (≥ 0,8 = grand effet).
- **Correction Holm** : nécessaire car on fait 3 tests (Baseline vs Blur, Baseline vs SAM3, Blur vs SAM3).

**Note méthodologique.** Avec n=8, la plus petite p-value possible d'un test de Wilcoxon
bilatéral est $p_{\min}=2/2^{8}\approx 0{,}008$, ce qui autorise pleinement le seuil
conventionnel $p<0{,}05$. Les conclusions sont reportées en combinant p-value, taille d'effet
(Cohen's $d_z$) et cohérence directionnelle sur les graines.


In [ ]:
# Test omnibus de Friedman : y a-t-il au moins une difference parmi les 3 pipelines ?
fr_f1 = sa.friedman_test(results, metric="test_macro_f1")
fr_acc = sa.friedman_test(results, metric="test_accuracy")

print("=== Friedman test (test omnibus) ===")
print(f"Macro-F1 : statistique = {fr_f1['friedman_statistic']:.3f}, "
      f"p-value = {fr_f1['friedman_pvalue']:.4f}")
print(f"Accuracy : statistique = {fr_acc['friedman_statistic']:.3f}, "
      f"p-value = {fr_acc['friedman_pvalue']:.4f}")


In [ ]:
# Comparaisons appariees pair-wise sur Macro-F1
pw = sa.all_pairwise_comparisons(results, metric="test_macro_f1", correction="holm")
pw["sig"] = pw["wilcoxon_pvalue_corrected"].apply(sa.significance_marker)

# Mise en forme lisible
disp = pw[[
    "pipeline_a", "pipeline_b", "mean_a", "mean_b", "mean_diff",
    "wilcoxon_pvalue", "wilcoxon_pvalue_corrected", "cohens_dz", "sig"
]].copy()
for col in ("mean_a", "mean_b", "mean_diff"):
    disp[col] = (disp[col] * 100).round(3)
for col in ("wilcoxon_pvalue", "wilcoxon_pvalue_corrected", "cohens_dz"):
    disp[col] = disp[col].round(4)
print("Comparaisons pair-wise sur Macro-F1 (correction Holm)")
print(disp.to_string(index=False))

pw.to_csv(TAB_DIR / "pairwise_comparisons_macro_f1.csv", index=False)


### Visualisation : différences appariées par seed

Le graphique en parallèle (parallel-coordinates) montre, pour chaque seed, l'évolution du
Macro-F1 d'un pipeline à l'autre. Si toutes les lignes vont dans le même sens, l'effet est
cohérent ; si elles se croisent, l'effet est instable selon le seed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, metric, title in zip(
    axes,
    ["test_macro_f1", "test_accuracy"],
    ["Macro-F1", "Accuracy"],
):
    long_metric = sa.long_format_metrics(results)
    sub = long_metric[long_metric["metric"] == metric]
    pivot = sub.pivot(index="seed", columns="pipeline", values="value")[list(sa.PIPELINES.keys())]
    for seed, row in pivot.iterrows():
        ax.plot(range(len(row)), row.values, marker="o", alpha=0.7, label=f"seed {seed}")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=15)
    ax.set_title(f"{title} — trajectoires par seed")
    ax.set_ylabel(metric.replace("_", " "))
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "parallel_coords_seeds.png")
plt.show()


In [ ]:
# Distribution des differences appariees (Macro-F1)
fig, ax = plt.subplots(figsize=(8, 5))
pairs_to_plot = [("baseline", "sam3_cnn"), ("baseline", "blur_light"), ("blur_light", "sam3_cnn")]
positions = []
for i, (a, b) in enumerate(pairs_to_plot):
    cmp = sa.paired_comparison(results, a, b, metric="test_macro_f1")
    diffs = np.array(cmp["diff_per_seed"]) * 100  # en points de %
    ax.scatter([i] * len(diffs), diffs, color="black", alpha=0.7, zorder=3)
    ax.hlines(diffs.mean(), i - 0.2, i + 0.2,
              colors="red", linewidth=2, zorder=2,
              label="moyenne" if i == 0 else None)
    positions.append(f"{b}\n-\n{a}")
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_xticks(range(len(positions)))
ax.set_xticklabels(positions)
ax.set_ylabel(r"$\Delta$ Macro-F1 (points de %)")
ax.set_title("Differences appariees par seed (Macro-F1)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "paired_differences.png")
plt.show()


## 3. Analyse par classe

Les métriques globales (Macro-F1, accuracy) peuvent **masquer** des effets contraires :
SAM3 peut aider sur certaines classes et nuire sur d'autres, et la moyenne en sortir neutre.
On regarde donc le F1 **par classe**, en se concentrant sur les classes identifiées comme
difficiles dans le rapport 2 :

- `tomato_verticulium_wilt` (faible support, symptômes diffus)
- `tomato_leaf_blight` (visuellement proche d'autres classes)
- `tomato_septoria_leaf_spot` (ambigue visuellement classes à nécroses foliaires)
- `tomato_yellow_leaf_curl_virus` (stades précoces ambigus)


In [ ]:
per_class = sa.per_class_summary(results)
per_class_pivot = per_class.pivot(index="classe", columns="pipeline", values="f1_mean")
per_class_pivot = per_class_pivot[list(sa.PIPELINES.keys())]
per_class_pivot["delta_sam_vs_baseline"] = per_class_pivot["sam3_cnn"] - per_class_pivot["baseline"]
per_class_pivot["delta_blur_vs_baseline"] = per_class_pivot["blur_light"] - per_class_pivot["baseline"]
per_class_pivot = per_class_pivot.sort_values("delta_sam_vs_baseline")
display_pivot = (per_class_pivot * 100).round(2)
display_pivot


In [ ]:
# Heatmap : F1 par classe x pipeline
fig, ax = plt.subplots(figsize=(8, 7))
heat_data = per_class_pivot[list(sa.PIPELINES.keys())] * 100
sns.heatmap(heat_data, annot=True, fmt=".2f", cmap="YlGnBu",
            cbar_kws={"label": "F1 moyen (%)"}, ax=ax,
            linewidths=0.4, linecolor="white")
ax.set_title("F1 par classe et par pipeline (moyenne sur 8 seeds)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_f1_par_classe.png")
plt.show()


In [ ]:
# Barplot des deltas par classe : SAM3 vs baseline
fig, ax = plt.subplots(figsize=(9, 6))
deltas = per_class_pivot["delta_sam_vs_baseline"] * 100
colors = ["#06A77D" if d > 0 else "#A23B72" for d in deltas]
ax.barh(deltas.index, deltas.values, color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(r"$\Delta$ F1 (points de %) - SAM3 - Baseline")
ax.set_title("Effet de SAM3 par classe (vert : amelioration, rose : degradation)")

# Marquer les classes difficiles
for i, classe in enumerate(deltas.index):
    if classe in sa.DIFFICULT_CLASSES:
        ax.text(deltas.values[i], i, "  *",
                va="center", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(FIG_DIR / "deltas_par_classe_sam3.png")
plt.show()
print("(*) classes identifiees comme difficiles dans le rapport 2.")


In [ ]:
# Test Wilcoxon par classe : Baseline vs SAM3
per_class_test = sa.per_class_paired_test(results, "baseline", "sam3_cnn")
per_class_test["sig"] = per_class_test["wilcoxon_pvalue"].apply(sa.significance_marker)
per_class_test["difficile"] = per_class_test["classe"].isin(sa.DIFFICULT_CLASSES)
disp = per_class_test.copy()
for col in ("f1_mean_a", "f1_mean_b", "delta_mean", "delta_std"):
    disp[col] = (disp[col] * 100).round(2)
disp["wilcoxon_pvalue"] = disp["wilcoxon_pvalue"].round(4)
print("Test Wilcoxon par classe : Baseline (a) vs SAM3 (b) - delta = b - a")
print(disp.to_string(index=False))
per_class_test.to_csv(TAB_DIR / "per_class_baseline_vs_sam3.csv", index=False)


**Hypothèse à tester pour le rapport :** SAM3 devrait aider sur les classes où **le fond
parasite l'apprentissage** (verticillium_wilt, où la zone malade est diffuse et mêlée au
contexte) et **ne pas changer grand-chose** sur les classes à symptômes très localisés
(septoria_leaf_spot, leaf_mold). Le tableau ci-dessus permet de valider ou invalider cette
hypothèse classe par classe.


## 4. Analyse résiduelle (matrices de confusion + accord/désaccord)

Cette section répond aux questions :

1. **Matrices de confusion agrégées** : où chaque pipeline se trompe-t-il systématiquement ?
   Les confusions sont-elles **les mêmes** entre Baseline et SAM3, ou bien SAM3 **déplace**-t-il
   les erreurs vers d'autres classes ?
2. **Accord / désaccord** : sur le même test set, combien d'images les deux pipelines
   classent-ils correctement ensemble (`both correct`), faussement ensemble (`both wrong`),
   ou différemment (`only A correct`, `only B correct`) ? Cette ventilation montre si SAM3
   apporte une **information complémentaire** à Baseline (cas favorable à un ensembling) ou
   s'il **redistribue** les erreurs sans en réduire le nombre.

### Précisions techniques

- **Re-prédiction** : on recharge chaque `best_resnet50_tomato.pth` (15 checkpoints) et on
  re-prédit sur le test set avec la même `transform` qu'à l'entraînement (`Resize 256x256`
  + normalisation ImageNet, **sans blur** — le blur du pipeline `blur_light` n'agit qu'au train).
- **Cache** : les prédictions sont sauvegardées en `.npz` dans `predictions_cache/`. Première
  exécution ~20–30 min sur GPU, ré-exécutions ~10 s.
- **Alignement Branche A vs B** : Baseline et SAM3 évaluent sur des dossiers différents
  (`tomato_final/test` en `.jpg` vs `tomato_segment_dataset/test` en `.png` avec suffixe `_crop`).
  La fonction `align_by_filename` du module gère automatiquement ce mapping :
  `IMG-20240724-WA0038.jpg` ↔ `IMG-20240724-WA0038_crop.png`. Le diagnostic effectué
  confirme une bijection parfaite (aucune image perdue par la segmentation).


In [ ]:
# A activer manuellement (long sur GPU la premiere fois ; cache ensuite)
RUN_RESIDUAL = True

# Adapter ces chemins a ton arborescence locale.
# verifie bien que les chemins existent avant de lancer.
TEST_DIR_BY_PIPELINE = {
    "baseline":   Path("/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/data/dataset_utilise/tomato_final/test"),
    "blur_light": Path("/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/data/dataset_utilise/tomato_final/test"),
    "sam3_cnn":   Path("/mnt/c/Users/LARIS-1/Desktop/DJMA/FINAL/data/dataset_utilise/tomato_segment_dataset/test"),
}

if RUN_RESIDUAL:
    # Sanity-check des chemins
    for p, d in TEST_DIR_BY_PIPELINE.items():
        if not d.exists():
            print(f"  ATTENTION : {p} -> {d} introuvable")
        else:
            n_classes = len([x for x in d.iterdir() if x.is_dir()])
            n_files = sum(1 for x in d.rglob('*') if x.is_file())
            print(f"  OK : {p:12s} -> {d}  ({n_classes} classes, {n_files} fichiers)")


In [ ]:
# Re-prediction (longue la premiere fois, instantanee ensuite grace au cache)
if RUN_RESIDUAL:
    confusion_results = sa.aggregate_confusion(
        results, TEST_DIR_BY_PIPELINE,
        cache_dir=CACHE_DIR,   # cache .npz par (pipeline, seed)
    )
    print("OK : predictions calculees / chargees depuis le cache.")
    for pipe, data in confusion_results.items():
        n_test = len(data["predictions_per_seed"][0]["y_true"])
        print(f"  {pipe:12s} : {n_test} images de test")


### 4.1 Matrices de confusion agrégées (moyenne sur 8 seeds, normalisées par ligne)

Une cellule (i, j) de la matrice normalisée représente le **recall conditionnel** : la
proportion d'images de la classe vraie i qui sont prédites en classe j. Diagonale forte
= bon pipeline.


In [ ]:
if RUN_RESIDUAL:
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    for ax, (pipe, data) in zip(axes, confusion_results.items()):
        cm = data["confusion_norm"]
        # Labels raccourcis pour la lisibilite
        short = [c.replace("tomato_", "").replace("_", " ")[:18] for c in data["classes"]]
        sns.heatmap(cm * 100, annot=True, fmt=".0f", cmap="Blues",
                    xticklabels=short, yticklabels=short,
                    ax=ax, cbar_kws={"label": "% par ligne"}, linewidths=0.3,
                    annot_kws={"fontsize": 7})
        ax.set_title(f"{PIPE_LABELS[pipe]}\n(moyenne 8 seeds)")
        ax.set_xlabel("Predit")
        ax.set_ylabel("Vrai" if pipe == "baseline" else "")
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "confusion_matrices_aggregees.png")
    plt.show()


### 4.2 Différence des matrices de confusion : SAM3 - Baseline

Cette figure est **la plus informative** pour le rapport : elle dit où SAM3 corrige Baseline
(bleu) et où SAM3 introduit de nouvelles erreurs (rouge).

- Une **diagonale bleue** signifie que SAM3 améliore le recall sur cette classe.
- Une **case rouge hors-diagonale** signifie que SAM3 confond davantage ces deux classes que Baseline.

Ce type de figure est ce que David appelle l'« **analyse résiduelle** » : on regarde la
**distribution des erreurs**, pas juste leur quantité totale.


In [ ]:
if RUN_RESIDUAL:
    delta_cm = (confusion_results["sam3_cnn"]["confusion_norm"]
                - confusion_results["baseline"]["confusion_norm"]) * 100
    classes = confusion_results["baseline"]["classes"]
    short = [c.replace("tomato_", "").replace("_", " ") for c in classes]

    fig, ax = plt.subplots(figsize=(11, 9))
    vmax = max(abs(np.nanmin(delta_cm)), abs(np.nanmax(delta_cm)))
    sns.heatmap(delta_cm, annot=True, fmt=".1f", cmap="RdBu_r", center=0,
                vmin=-vmax, vmax=vmax,
                xticklabels=short, yticklabels=short,
                cbar_kws={"label": r"$\Delta$ (pts de %) - SAM3 - Baseline"}, ax=ax,
                annot_kws={"fontsize": 8})
    ax.set_title("Difference des matrices de confusion : SAM3 - Baseline\n"
                 "(bleu : SAM3 corrige | rouge : SAM3 introduit l'erreur)")
    ax.set_xlabel("Predit")
    ax.set_ylabel("Vrai")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "confusion_diff_sam3_vs_baseline.png")
    plt.show()


In [ ]:
from torchvision import datasets
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

def get_test_paths(test_dir):
    ds = datasets.ImageFolder(test_dir)
    paths = [p for p, _ in ds.samples]
    labels = [y for _, y in ds.samples]
    classes = ds.classes
    return paths, np.array(labels), classes
def show_disagreement_examples(
    confusion_results,
    TEST_DIR_BY_PIPELINE,
    seed_idx=0,
    mode="baseline_correct_sam3_wrong",
    max_images=12,
):
    """
    mode:
    - baseline_correct_sam3_wrong : SAM3 introduit une erreur
    - baseline_wrong_sam3_correct : SAM3 corrige une erreur
    - both_wrong                  : les deux modèles se trompent
    """

    # Test set baseline : images brutes
    base_paths, y_true_base, classes = get_test_paths(TEST_DIR_BY_PIPELINE["baseline"])

    # Prédictions pour une seed donnée
    y_true = confusion_results["baseline"]["predictions_per_seed"][seed_idx]["y_true"]
    y_pred_base = confusion_results["baseline"]["predictions_per_seed"][seed_idx]["y_pred"]
    y_pred_sam = confusion_results["sam3_cnn"]["predictions_per_seed"][seed_idx]["y_pred"]

    assert np.array_equal(y_true, y_true_base), "Attention : ordre du test set incohérent."

    if mode == "baseline_correct_sam3_wrong":
        idxs = np.where((y_pred_base == y_true) & (y_pred_sam != y_true))[0]
        title = "Images où Baseline est correct et SAM3 se trompe"

    elif mode == "baseline_wrong_sam3_correct":
        idxs = np.where((y_pred_base != y_true) & (y_pred_sam == y_true))[0]
        title = "Images où SAM3 corrige la Baseline"

    elif mode == "both_wrong":
        idxs = np.where((y_pred_base != y_true) & (y_pred_sam != y_true))[0]
        title = "Images où les deux modèles se trompent"

    else:
        raise ValueError("mode inconnu")

    idxs = idxs[:max_images]

    if len(idxs) == 0:
        print("Aucun exemple trouvé pour ce mode.")
        return

    ncols = 4
    nrows = int(np.ceil(len(idxs) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, idxs):
        img = Image.open(base_paths[idx]).convert("RGB")

        true_label = classes[y_true[idx]].replace("tomato_", "")
        pred_base = classes[y_pred_base[idx]].replace("tomato_", "")
        pred_sam = classes[y_pred_sam[idx]].replace("tomato_", "")

        ax.imshow(img)
        ax.axis("off")
        ax.set_title(
            f"Vrai: {true_label}\n"
            f"Baseline: {pred_base}\n"
            f"SAM3: {pred_sam}",
            fontsize=9
        )

    for ax in axes[len(idxs):]:
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"examples_{mode}_seed{seed_idx}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Voir les cas où SAM3 introduit une erreur
seed_idx = 0
print(f"Seed utilisée : {sa.SEEDS[seed_idx]}")

show_disagreement_examples(
    confusion_results,
    TEST_DIR_BY_PIPELINE,
    seed_idx=seed_idx,
    mode="baseline_correct_sam3_wrong",
    max_images=12,
)

In [ ]:
# Voir les cas où SAM3 corrige la baseline
show_disagreement_examples(
    confusion_results,
    TEST_DIR_BY_PIPELINE,
    seed_idx=0,
    mode="baseline_wrong_sam3_correct",
    max_images=12,
)

In [ ]:
# Voir les cas où les deux modèles échouent
show_disagreement_examples(
    confusion_results,
    TEST_DIR_BY_PIPELINE,
    seed_idx=0,
    mode="both_wrong",
    max_images=12,
)

### 4.3 Accord / désaccord par image (Baseline vs SAM3)

Sur les images **présentes dans les deux datasets** (alignées par filename), on ventile
les prédictions des deux pipelines en 4 catégories :

| Catégorie         | Baseline | SAM3   | Interprétation                                         |
|-------------------|----------|--------|---------------------------------------------------------|
| `both_correct`    | ✓        | ✓      | Cas faciles, les deux réussissent                       |
| `both_wrong`      | ✗        | ✗      | Cas durs, les deux échouent (seg ne suffit pas à corriger) |
| `only_a_correct`  | ✓        | ✗      | SAM3 dégrade : la segmentation a perdu de l'info utile  |
| `only_b_correct`  | ✗        | ✓      | SAM3 corrige : isoler la feuille a aidé                  |

L'**information complémentaire** apportée par SAM3 = `only_b_correct`. Si elle est
significative (par exemple > 2 % du test set), un **ensembling A+B** peut faire mieux
que B seul, ce qui est une piste à mentionner dans la conclusion.


In [ ]:
if RUN_RESIDUAL:
    pa = confusion_results["baseline"]["predictions_per_seed"]
    pb = confusion_results["sam3_cnn"]["predictions_per_seed"]
    agreement = sa.agreement_analysis(pa, pb, align=True)
    df_agree = agreement["per_seed"]

    print(f"Alignement par filename : {agreement['n_aligned_per_seed']} images par seed")
    print(f"(Tomato_final test = {len(pa[0]['filenames'])} images, "
          f"Tomato_segment test = {len(pb[0]['filenames'])} images)\n")

    disp = df_agree.copy()
    for col in ("accuracy_a", "accuracy_b", "agreement_rate",
                "both_correct", "both_wrong", "only_a_correct", "only_b_correct"):
        disp[col] = (disp[col] * 100).round(2)
    print("Accord / desaccord Baseline vs SAM3 (% sur test aligne)")
    print(disp.to_string(index=False))

    df_agree.to_csv(TAB_DIR / "agreement_baseline_vs_sam3.csv", index=False)


In [ ]:
# Visualisation : empilement 'both correct / both wrong / only A / only B' par seed
if RUN_RESIDUAL:
    df_agree = agreement["per_seed"].copy()
    seeds_lbl = [f"seed {sa.SEEDS[i]}" for i in df_agree["seed_idx"]]
    cats = ["both_correct", "only_a_correct", "only_b_correct", "both_wrong"]
    cat_labels = [
        "Both correct",
        "Only A correct (Baseline)",
        "Only B correct (SAM3)",
        "Both wrong",
    ]
    cat_colors = ["#2E86AB", "#F4A261", "#06A77D", "#A23B72"]
    fig, ax = plt.subplots(figsize=(9, 5))
    bottoms = np.zeros(len(df_agree))

    for cat, lbl, c in zip(cats, cat_labels, cat_colors):
        vals = (df_agree[cat] * 100).to_numpy()

        bars = ax.bar(
            seeds_lbl,
            vals,
            bottom=bottoms,
            color=c,
            label=lbl,
            edgecolor="white",
            linewidth=0.5
        )

        # Ajout des pourcentages 
        for i, (bar, val) in enumerate(zip(bars, vals)):
            if val > 2:  # évite les segments trop petits
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bottoms[i] + val / 2,
                    f"{val:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white"
                )

        bottoms += vals

    ax.set_ylabel("Pourcentage du test set aligné (%)")
    ax.set_title("Accord / désaccord Baseline vs SAM3 — par seed")

    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "agreement_stacked.png", dpi=300, bbox_inches="tight")
    plt.show()

### 4.4 Mêmes erreurs ou erreurs différentes ?

Pour répondre directement à la question « **est-ce que les erreurs sont les mêmes** » entre
les deux pipelines :


In [ ]:
if RUN_RESIDUAL:
    df = agreement["per_seed"]
    n_aligned_mean = np.mean(agreement["n_aligned_per_seed"])
    print(f"Moyenne sur {len(df)} seeds (n_aligne ~= {n_aligned_mean:.0f} images) :\n")
    print(f"  Taux d'accord global   : {df['agreement_rate'].mean()*100:.2f}%  "
          f"(images sur lesquelles les deux predisent la meme classe)")
    print(f"  Both correct           : {df['both_correct'].mean()*100:.2f}%")
    print(f"  Both wrong (memes !)   : {df['both_wrong'].mean()*100:.2f}%")
    print(f"  Only A correct         : {df['only_a_correct'].mean()*100:.2f}%  "
          f"(SAM3 degrade)")
    print(f"  Only B correct         : {df['only_b_correct'].mean()*100:.2f}%  "
          f"(SAM3 corrige)")

    # Ratio gain/perte : > 1 = SAM3 corrige plus qu'il ne degrade
    gain_loss = df['only_b_correct'].mean() / max(df['only_a_correct'].mean(), 1e-9)
    print(f"\n  Ratio (only_B_correct / only_A_correct) = {gain_loss:.2f}")
    print(f"  >>> {'SAM3 corrige plus qu il ne degrade' if gain_loss > 1 else 'SAM3 degrade plus qu il ne corrige'}")


### 4.5 Diagnose qualitative : quelles classes sont concernées ?

L'analyse précédente nous donne les pourcentages globaux d'accord/désaccord.
On veut maintenant savoir **quelles classes** sont surreprésentées dans chaque cas
de désaccord. Trois questions :

- **`both_wrong`** : sur quelles classes les deux pipelines échouent simultanément ?
  Ce sont les *cas durs intrinsèques* du dataset.
- **`only_a_correct`** (Baseline réussit, SAM3 échoue) : sur quelles classes SAM3 dégrade ?
  Ce sont les classes dont le contexte (fond, plante entière) apporte de l'information.
- **`only_b_correct`** (SAM3 réussit, Baseline échoue) : sur quelles classes SAM3 corrige ?
  Ce sont les classes où isoler la feuille a vraiment aidé.

Pour chaque cas, on agrège les images sur les 8 graines (une image peut être comptée
plusieurs fois si plusieurs graines la classent dans le même cas).


In [ ]:
# Analyse quantitative : distribution par classe pour chaque cas de desaccord
# Agrege sur toutes les graines (chaque image peut etre comptee plusieurs fois)

if RUN_RESIDUAL:
    pa_list = confusion_results["baseline"]["predictions_per_seed"]
    pb_list = confusion_results["sam3_cnn"]["predictions_per_seed"]
    classes = confusion_results["baseline"]["classes"]

    # Pour chaque seed, on aligne par filename et on collecte les indices
    # de chaque cas de desaccord, indexes par la classe vraie
    cases = {"both_wrong": [], "only_a_correct": [], "only_b_correct": []}

    for seed_idx in range(len(pa_list)):
        pa, pb, _ = sa.align_by_filename(pa_list[seed_idx], pb_list[seed_idx])
        ya, predsa = pa["y_true"], pa["y_pred"]
        yb, predsb = pb["y_true"], pb["y_pred"]
        a_correct = (predsa == ya)
        b_correct = (predsb == yb)
        cases["both_wrong"].append(np.where(~a_correct & ~b_correct)[0])
        cases["only_a_correct"].append(np.where(a_correct & ~b_correct)[0])
        cases["only_b_correct"].append(np.where(~a_correct & b_correct)[0])

    # Construire un tableau : classe vraie x cas, comptages agreges sur 8 seeds
    case_labels = {
        "both_wrong": "Both wrong (cas durs intrinsèques)",
        "only_a_correct": "Only A correct (SAM3 dégrade)",
        "only_b_correct": "Only B correct (SAM3 corrige)",
    }
    rows = []
    for case_name, idx_per_seed in cases.items():
        # Concatener tous les y_true des images concernees, sur toutes les seeds
        ya_all = np.concatenate([
            sa.align_by_filename(pa_list[s], pb_list[s])[0]["y_true"][idxs]
            for s, idxs in enumerate(idx_per_seed)
        ])
        for cls_idx, cls_name in enumerate(classes):
            n = int((ya_all == cls_idx).sum())
            rows.append({
                "cas": case_labels[case_name],
                "classe": cls_name.replace("tomato_", ""),
                "n_total_8seeds": n,
                "n_par_seed_moyen": n / 8.0,
            })
    df_cases = pd.DataFrame(rows)

    # Pivot : une ligne par classe, une colonne par cas
    pivot = df_cases.pivot(index="classe", columns="cas", values="n_par_seed_moyen")
    pivot["support_test"] = [
        sum(pa_list[0]["y_true"] == i) for i in range(len(classes))
    ]  # support reel dans le test set (par seed, identique entre seeds)
    pivot = pivot.round(1).sort_values("Only A correct (SAM3 dégrade)", ascending=False)

    print("Distribution par classe pour chaque cas de desaccord")
    print("(valeurs = nombre moyen d'images par graine)\n")
    print(pivot.to_string())

    df_cases.to_csv(TAB_DIR / "desaccord_par_classe.csv", index=False)
    pivot.to_csv(TAB_DIR / "desaccord_par_classe_pivot.csv")


In [ ]:
# Visualisation : 3 barplots horizontaux cote-a-cote

if RUN_RESIDUAL:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    case_colors = {
        "both_wrong": "#A23B72",
        "only_a_correct": "#F4A261",
        "only_b_correct": "#06A77D",
    }
    case_titles = {
        "both_wrong": "Both wrong\n(les 2 pipelines echouent)",
        "only_a_correct": "Only A correct\n(SAM3 degrade)",
        "only_b_correct": "Only B correct\n(SAM3 corrige)",
    }
    # Ordonner les classes par support pour rendre lisible
    classes_short = [c.replace("tomato_", "") for c in classes]
    support_per_class = np.array([
        sum(pa_list[0]["y_true"] == i) for i in range(len(classes))
    ])
    order_idx = np.argsort(-support_per_class)
    classes_ordered = [classes_short[i] for i in order_idx]

    # Marquer les classes difficiles avec un asterisque (*)
    # sa.DIFFICULT_CLASSES contient les noms complets type "tomato_verticulium_wilt"
    difficult_short = {c.replace("tomato_", "") for c in sa.DIFFICULT_CLASSES}
    yticklabels = [
        f"{c} *" if c in difficult_short else c
        for c in classes_ordered
    ]

    for ax, (case_name, color) in zip(axes, case_colors.items()):
        # Compter par classe sur toutes les seeds
        ya_all = np.concatenate([
            sa.align_by_filename(pa_list[s], pb_list[s])[0]["y_true"][idxs]
            for s, idxs in enumerate(cases[case_name])
        ])
        counts = np.array([(ya_all == i).sum() / 8 for i in range(len(classes))])
        counts_ordered = counts[order_idx]
        ax.barh(yticklabels, counts_ordered, color=color,
                edgecolor="black", linewidth=0.4)
        ax.set_title(case_titles[case_name])
        ax.set_xlabel("Nombre moyen d'images par graine")
        ax.invert_yaxis()
        # Annotations
        for i, v in enumerate(counts_ordered):
            if v > 0.5:
                ax.text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8)

    plt.suptitle("Distribution par classe des cas de desaccord (moyenne sur 8 graines)\n"
                 "(*) classes identifiees comme difficiles dans le rapport 2",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "desaccord_par_classe_barplots.png")
    plt.show()


### 4.6 Visualisation aléatoire d'images dans chaque cas

On tire **au hasard** quelques images dans chaque cas de désaccord, agrégées sur
toutes les graines, pour voir concrètement à quoi ressemblent ces erreurs.
Le tirage est reproductible (`np.random.seed(0)`).


In [ ]:
# Tirage aleatoire reproductible et visualisation

if RUN_RESIDUAL:
    from PIL import Image

    # Recuperer les chemins reels des images du test set baseline
    base_paths_full, _, classes_check = get_test_paths(TEST_DIR_BY_PIPELINE["baseline"])
    base_filenames = [Path(p).name for p in base_paths_full]

    # ATTENTION : align_by_filename retourne des CLES CANONIQUES (sans extension,
    # sans suffixe _crop). Pour retrouver le chemin sur disque, on doit indexer
    # le mapping par la meme cle canonique.
    fname_to_path = {
        sa._strip_suffixes(name): full
        for name, full in zip(base_filenames, base_paths_full)
    }
    print(f"Mapping construit : {len(fname_to_path)} chemins indexes par cle canonique")

    # Construit une liste (cas, key, true, pred_a, pred_b, seed)
    # en consolidant sur toutes les seeds, sans doublons sur (key, cas)
    consolidated = {"both_wrong": [], "only_a_correct": [], "only_b_correct": []}
    seen = {"both_wrong": set(), "only_a_correct": set(), "only_b_correct": set()}

    for seed_idx in range(len(pa_list)):
        pa, pb, _ = sa.align_by_filename(pa_list[seed_idx], pb_list[seed_idx])
        ya = pa["y_true"]
        predsa = pa["y_pred"]
        predsb = pb["y_pred"]
        keys = pa["filenames"]   # ce sont les cles canoniques (deja normalisees)
        for case_name, mask in [
            ("both_wrong",       (predsa != ya) & (predsb != ya)),
            ("only_a_correct",   (predsa == ya) & (predsb != ya)),
            ("only_b_correct",   (predsa != ya) & (predsb == ya)),
        ]:
            for i in np.where(mask)[0]:
                key = keys[i]
                if key in seen[case_name]:
                    continue
                seen[case_name].add(key)
                consolidated[case_name].append({
                    "key": key,
                    "true": classes[ya[i]].replace("tomato_", ""),
                    "pred_a": classes[predsa[i]].replace("tomato_", ""),
                    "pred_b": classes[predsb[i]].replace("tomato_", ""),
                    "seed_idx": seed_idx,
                })

    print("\nEffectif unique d\'images par cas (apres deduplication sur les 8 seeds) :")
    for c, lst in consolidated.items():
        print(f"  {c:18s} : {len(lst)} images uniques")
    print()

    # Tirage aleatoire reproductible et affichage
    rng = np.random.default_rng(0)

    def show_random_examples(case_name, n=12, ncols=4):
        items = consolidated[case_name]
        if len(items) == 0:
            print(f"Aucune image pour {case_name}.")
            return
        sel = rng.choice(len(items), size=min(n, len(items)), replace=False)
        sel_items = [items[i] for i in sel]

        nrows = int(np.ceil(len(sel_items) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.5 * nrows))
        axes = np.array(axes).reshape(-1)
        n_displayed = 0
        for ax, item in zip(axes, sel_items):
            path = fname_to_path.get(item["key"])
            if path is None:
                ax.set_title(f"Fichier introuvable : {item['key']}", fontsize=8)
                ax.axis("off")
                continue
            img = Image.open(path).convert("RGB")
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(
                f"Vrai: {item['true']}\n"
                f"Baseline: {item['pred_a']}\n"
                f"SAM3: {item['pred_b']}",
                fontsize=9,
            )
            n_displayed += 1
        for ax in axes[len(sel_items):]:
            ax.axis("off")

        title_map = {
            "both_wrong": "Both wrong : les deux pipelines echouent",
            "only_a_correct": "Only A correct : SAM3 degrade par rapport a Baseline",
            "only_b_correct": "Only B correct : SAM3 corrige Baseline",
        }
        plt.suptitle(
            title_map[case_name] + f" — tirage aleatoire de {n_displayed} images",
            fontsize=13, y=1.0,
        )
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"random_{case_name}.png", dpi=200)
        plt.show()

    show_random_examples("both_wrong", n=12)
    show_random_examples("only_a_correct", n=12)
    show_random_examples("only_b_correct", n=12)


## 5. Synthèse pour le rapport

À remplir une fois les cellules exécutées :

- **Effet global de SAM3** (Branche B vs A) : ΔMacro-F1 = ___ ± ___ pts ;
  Wilcoxon p = ___ ; Cohen's d_z = ___.
- **Conclusion à n=8** : significatif / tendance / non significatif (la limite p ≥ 0,008
  du Wilcoxon à n=8 autorise pleinement le seuil conventionnel p<0,05).
- **Effet par classe** : SAM3 améliore ___, dégrade ___, sans effet sur ___.
- **Hypothèse confirmée / infirmée** : SAM3 aide-t-il bien sur les classes à fond parasitant
  (`verticillium_wilt`) ?
- **Erreurs partagées vs spécifiques** : `both_wrong` = ___ % (cas durs intrinsèques),
  `only_b_correct` = ___ % (gain net de SAM3), `only_a_correct` = ___ % (perte due à SAM3).
- **Implications pour la suite** : si SAM3 dégrade certaines classes, les pistes sont
  (1) un meilleur prompt SAM3, (2) un fine-tuning de SAM3 (ou SPROUT), (3) un ensembling
  Branche A + Branche B (justifié si `only_b_correct` est non-trivial).

### Bonnes pratiques de reporting dans le rapport

- Toujours reporter **n** (= 8 seeds) à côté de tout p-value.
- Préférer **moyenne ± std + IC95 % bootstrap** plutôt que p-value seule.
- Pour les comparaisons multiples (3 paires), reporter la **p corrigée Holm** à côté de la p brute.
- Mentionner explicitement la **limite n=8 → p_min ≈ 0,008** dans la sous-section méthodes.
- Inclure le tableau **par-seed** en annexe (transparence totale).
